# Fermi-GBM Time-Resolved Spectral Analysis Pipeline
Each cell corresponds to one pipeline stage — run them top to bottom, or re-run any single stage independently.

## Cell 1 — Imports & Setup

In [16]:
import os
import logging
import numpy as np
import matplotlib
matplotlib.use('Agg')   # swap to 'inline' if running interactively
import matplotlib.pyplot as plt
from astropy.table import Table
import astropy.units as u

from threeML import (
    silence_logs, download_GBM_trigger_data, TimeSeriesBuilder,
    OGIPLike, DataList, Model, PointSource, BayesianAnalysis, plot_spectra,
)
from threeML.catalogs import FermiGBMBurstCatalog as FermiGBMBurst
from astromodels import Cutoff_powerlaw, Band, Log_uniform_prior, Truncated_gaussian

silence_logs()
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('GBM_pipeline')
print('Imports OK')


Imports OK


## Cell 2 — Configuration

In [2]:
# ── Edit this block only ────────────────────────────────────────────
CONFIG = dict(
    trigger_name   = 'bn140304557',   # GBM trigger ID  (no 'GRB' prefix)
    grb_name       = 'GRB140304557',  # catalog key      (with 'GRB' prefix)
    ra             = 30.653,           # RA  of the burst [deg]  — update!
    dec            =  33.480,           # Dec of the burst [deg]  — update!

    # detector selection
    max_rov        = 60.0,            # keep detectors with ROV < this [deg]

    # Bayesian-blocks settings
    bb_t_start     = -10.0,           # start of BB window [s relative to trigger]
    bb_t_stop      =  50.0,           # end   of BB window
    bb_p0          =  0.01,           # false-alarm probability for BB
    min_bin_width  =  0.05,           # seconds; narrower bins are merged away

    # spectral energy ranges
    nai_range      = '8-33, 37-900',  # keV  (gap skips K-edge ~34-36 keV)
    bgo_range      = '200-40000',     # keV

    # CutoffPL-function priors
    priors = dict(
        K     = dict(lower=1e-5, upper=1e1),
        index = dict(lower=-2.0, upper=1.0,  mu=-1.3,  sigma=0.5),
        xc    = dict(lower=10.0, upper=1000, mu=300.0, sigma=200.0)
    ),

    # MultiNest settings
    n_live_points  = 100,             # increase to >=1000 for publication quality

    # output
    outdir         = 'gbm_pipeline_output',
)

# ── Create output directories ─────────────────────────────────────────
PATHS = {
    'root':    CONFIG['outdir'],
    'spectra': os.path.join(CONFIG['outdir'], 'spectra'),
    'corners': os.path.join(CONFIG['outdir'], 'corners'),
    'lc':      os.path.join(CONFIG['outdir'], 'lightcurves'),
}
for d in PATHS.values():
    os.makedirs(d, exist_ok=True)

print('Config loaded. Output directories created at:', CONFIG['outdir'])


Config loaded. Output directories created at: gbm_pipeline_output


## Cell 3 — Helper Function Definitions

In [4]:
def pick_reference_detector(detectors):
    """Prefer n3 as BB reference; fall back to first NaI, then first any."""
    nai = [d for d in detectors if d.startswith('n')]
    if 'n3' in nai: return 'n3'
    return nai[0] if nai else detectors[0]


def build_cutoffpl_model(ra, dec, priors):
    """Return a fresh Band-function Model with the configured priors."""
    band  = Cutoff_powerlaw()
    model = Model(PointSource('GBM', ra, dec, spectral_shape=band))
    p = priors
    model.GBM.spectrum.main.Cutoff_powerlaw.K.prior = Log_uniform_prior(
        lower_bound=p['K']['lower'], upper_bound=p['K']['upper'])
    model.GBM.spectrum.main.Cutoff_powerlaw.index.prior = Truncated_gaussian(
        lower_bound=p['index']['lower'], upper_bound=p['index']['upper'],
        mu=p['index']['mu'], sigma=p['index']['sigma'])
    model.GBM.spectrum.main.Cutoff_powerlaw.xc.prior = Truncated_gaussian(
        lower_bound=p['xc']['lower'], upper_bound=p['xc']['upper'],
        mu=p['xc']['mu'], sigma=p['xc']['sigma'])
    return model


print('Helper functions defined.')


Helper functions defined.


## Cell 4 — Detector Selection (ROV < 60°)

In [8]:
log.info('Querying Fermi-GBM catalog for %s ...', CONFIG['grb_name'])

cat = FermiGBMBurst()
cat.query_sources(CONFIG['grb_name'])
grb_info = cat.get_detector_information()[CONFIG['grb_name']]

# ── Fetch catalog source / background intervals ───────────────────────
source_interval     = grb_info['source']['fluence']
background_interval = grb_info['background']['full']
log.info('Source interval    : %s', source_interval)
log.info('Background interval: %s', background_interval)

# ── Manually Specify Detectors ────────────────────────────────────────
# Valid NaI detectors: 'n0' through 'n9', 'na', 'nb'
# Valid BGO detectors: 'b0', 'b1'
detectors = ['n3', 'n4', 'b0']  # <-- Change these to your preferred detectors

log.info('Manually selected detectors done: %s', detectors)


# ── Quick sanity check ────────────────────────────────────────────────
assert len(detectors) > 0, 'No detectors selected. Please add them to the list.'
assert any(d.startswith('n') for d in detectors), 'Need at least one NaI detector for the fit.'


12:06:37  INFO      Querying Fermi-GBM catalog for GRB140304557 ...
12:06:43  INFO      Source interval    : -5.376000-25.856000
12:06:43  INFO      Background interval: -40.830000--11.250000,41.670000-97.920000
12:06:43  INFO      Manually selected detectors done: ['n3', 'n4', 'b0']


## Cell 5 — Download GBM Data

In [9]:
log.info('Downloading TTE / CSPEC / RSP for: %s', detectors)
dload = download_GBM_trigger_data(CONFIG['trigger_name'], detectors=detectors)

# ── Verify all expected file types are present ────────────────────────
for det in detectors:
    for ftype in ('tte', 'cspec', 'rsp'):
        assert ftype in dload[det], f'Missing {ftype} file for detector {det}'

print('Download complete. Files available for:', list(dload.keys()))


12:06:47  INFO      Downloading TTE / CSPEC / RSP for: ['n3', 'n4', 'b0']


Download complete. Files available for: ['n3', 'n4', 'b0']


## Cell 6 — Build Time Series & Fit Backgrounds

In [10]:
time_series = {}

for det in detectors:
    log.info('Processing detector %s ...', det)

    # ── Step 6a: fit background polynomial on CSPEC ───────────────────
    ts_cspec = TimeSeriesBuilder.from_gbm_cspec_or_ctime(
        det,
        cspec_or_ctime_file = dload[det]['cspec'],
        rsp_file            = dload[det]['rsp'],
    )
    bkg_segs = [s.strip() for s in background_interval.split(',')]
    ts_cspec.set_background_interval(*bkg_segs)

    bkg_file = os.path.join(PATHS['root'], f'{det}_bkg.h5')
    ts_cspec.save_background(bkg_file, overwrite=True)
    log.info('  Background saved to %s', bkg_file)

    # ── Step 6b: load TTE with restored background ────────────────────
    ts_tte = TimeSeriesBuilder.from_gbm_tte(
        det,
        tte_file           = dload[det]['tte'],
        rsp_file           = dload[det]['rsp'],
        restore_background = bkg_file,
    )
    ts_tte.set_active_time_interval(source_interval)
    time_series[det] = ts_tte
    log.info('  TTE loaded and active interval set.')

print('Time series built for:', list(time_series.keys()))

# ── Quick sanity: view lightcurve of first NaI ────────────────────────
first_nai = next(d for d in detectors if d.startswith('n'))
fig = time_series[first_nai].view_lightcurve(-40, 100)
fig.savefig(os.path.join(PATHS['lc'], f'{first_nai}_raw_lc.png'), dpi=100, bbox_inches='tight')
plt.close('all')
print(f'Raw lightcurve saved for {first_nai}')


12:08:35  INFO      Processing detector n3 ...

WARNING RuntimeWarning: invalid value encountered in divide



Loading PHAII Spectra:   0%|                                                                                | …


WARNING RuntimeWarning: invalid value encountered in divide



Finding best polynomial Order:   0%|                                                                          …

Fitting GBM_NAI_03 background:   0%|                                                                         |…

...


WARNING RuntimeWarning: invalid value encountered in divide



Finding best polynomial Order:   0%|                                                                          …

Fitting GBM_BGO_00 background:   0%|                                                                         |…

12:09:40  INFO        Background saved to gbm_pipeline_output/b0_bkg.h5
12:09:40  INFO        TTE loaded and active interval set.


Time series built for: ['n3', 'n4', 'b0']
Raw lightcurve saved for n3


## Cell 7 — Bayesian-Block Time Binning

In [11]:
ref_det = pick_reference_detector(detectors)
ts_ref  = time_series[ref_det]
log.info('Reference detector for BB: %s', ref_det)

# ── Run Bayesian blocks ────────────────────────────────────────────────
ts_ref.create_time_bins(
    CONFIG['bb_t_start'], CONFIG['bb_t_stop'],
    method='bayesblocks', use_background=True, p0=CONFIG['bb_p0'],
)

print('Raw BB bins:')
print('  starts:', ts_ref.bins.starts)
print('  stops :', ts_ref.bins.stops)
print('  widths:', ts_ref.bins.widths)

fig_raw = ts_ref.view_lightcurve(use_binner=True)
fig_raw.savefig(os.path.join(PATHS['lc'], f'{ref_det}_bb_raw.png'), dpi=120, bbox_inches='tight')
plt.close('all')


12:10:46  INFO      Reference detector for BB: n3


Raw BB bins:
  starts: [-9.999761998653412, -6.61941796541214, 7.057295024394989, 11.157336056232452]
  stops : [-6.61941796541214, 7.057295024394989, 11.157336056232452, 49.99896001815796]
  widths: [ 3.38034403 13.67671299  4.10004103 38.84162396]


## Cell 8 — Merge Sub-Threshold Bins

In [12]:
min_w = CONFIG['min_bin_width']

bad_bins = {i for i, w in enumerate(ts_ref.bins.widths) if w < min_w}
print(f'Bins narrower than {min_w} s (to be merged): {sorted(bad_bins)}')

edges = [ts_ref.bins.starts[0]]
for i, b in enumerate(ts_ref.bins):
    if i not in bad_bins:
        edges.append(b.stop)

bb_starts = np.array(edges[:-1])
bb_stops  = np.array(edges[1:])
n_bins    = len(bb_starts)

ts_ref.create_time_bins(bb_starts, bb_stops, method='custom')

print(f'Clean bins ({n_bins} total):')
print('  starts:', bb_starts)
print('  stops :', bb_stops)
print('  widths:', bb_stops - bb_starts)

fig_clean = ts_ref.view_lightcurve(use_binner=True)
fig_clean.savefig(os.path.join(PATHS['lc'], f'{ref_det}_bb_clean.png'), dpi=120, bbox_inches='tight')
plt.close('all')

assert n_bins > 0, 'Zero bins after merging — check bb_t_start / bb_t_stop or p0'
print(f'\nReady to fit {n_bins} time bin(s).')


Bins narrower than 0.05 s (to be merged): []
Clean bins (4 total):
  starts: [-9.999762   -6.61941797  7.05729502 11.15733606]
  stops : [-6.61941797  7.05729502 11.15733606 49.99896002]
  widths: [ 3.38034403 13.67671299  4.10004103 38.84162396]

Ready to fit 4 time bin(s).


## Cell 9 — Sync Detectors & Build Time-Resolved Plugins

In [13]:
time_resolved_plugins = {}   # {det: [plugin_bin_0, plugin_bin_1, ...]}

for det in detectors:
    ts = time_series[det]
    ts.read_bins(ts_ref)                         # sync to reference bins
    plugin_list = ts.to_spectrumlike(from_bins=True)
    time_resolved_plugins[det] = plugin_list
    log.info('%s  ->  %d time-bin plugins', det, len(plugin_list))

# ── Sanity: all detectors should have the same number of bins ─────────
bin_counts = {d: len(v) for d, v in time_resolved_plugins.items()}
print('Plugin counts per detector:', bin_counts)
assert len(set(bin_counts.values())) == 1, 'Bin count mismatch across detectors!'
print('All detectors in sync. Ready for spectral fitting.')


Creating plugins:   0%|                                                                                       …

12:11:43  INFO      n3  ->  4 time-bin plugins


Creating plugins:   0%|                                                                                       …

12:11:43  INFO      n4  ->  4 time-bin plugins


Creating plugins:   0%|                                                                                       …

12:11:45  INFO      b0  ->  4 time-bin plugins


Plugin counts per detector: {'n3': 4, 'n4': 4, 'b0': 4}
All detectors in sync. Ready for spectral fitting.


## Cell 10 — Time-Resolved Spectral Fitting (CutoffPLModel)

In [ ]:
PARAM_NAMES = [
    'B_K',     'B_K_p_err',     'B_K_n_err',     'B_K_avg_err',
    'B_alpha', 'B_alpha_n_err', 'B_alpha_p_err',  'B_alpha_avg_err',
    'E_c',  'E_c_n_err',  'E_c_p_err',   'E_c_avg_err'
]

fit_rows = []

for i in range(n_bins):
    log.info('==== Time bin %d / %d ====', i + 1, n_bins)

    # ── Assemble per-detector plugins for this bin ────────────────────
    det_plugins = []
    for det in detectors:
        p = time_resolved_plugins[det][i]
        if det.startswith('b'):
            p.set_active_measurements(CONFIG['bgo_range'])
        else:
            ranges = [r.strip() for r in CONFIG['nai_range'].split(',')]
            p.set_active_measurements(*ranges)
        det_plugins.append(p)

    data_list = DataList(*det_plugins)

    # ── Fresh Band model per bin ──────────────────────────────────────
    model = build_cutoffpl_model(CONFIG['ra'], CONFIG['dec'], CONFIG['priors'])

    # ── Bayesian fit with MultiNest ───────────────────────────────────
    bayes = BayesianAnalysis(model, data_list)
    bayes.set_sampler('multinest')
    bayes.sampler.setup(n_live_points=CONFIG['n_live_points'])
    bayes.sample()

    # ── νFν spectrum plot ─────────────────────────────────────────────
    fig_spec = plot_spectra(
        bayes.results,
        ene_min=1.0, ene_max=1e5, num_ene=200,
        contour_colors='white',
        contour_style_kwargs=dict(alpha=0.2),
        use_components=True, components_to_use=['total'],
        flux_unit='keV2/(cm2 s keV)',
    )
    ax = plt.gca()
    ax.set_xlabel('Energy (keV)', fontsize=16)
    ax.set_ylabel(r'$\nu\,F_\nu\;[\mathrm{keV\,cm^{-2}\,s^{-1}}]$', fontsize=16)
    ax.set_title(f"{CONFIG['trigger_name']}  bin {i+1}/{n_bins}  "
                 f"[{bb_starts[i]:.3f}, {bb_stops[i]:.3f}] s", fontsize=13)
    fig_spec.savefig(
        os.path.join(PATHS['spectra'], f'band_spec_bin{i+1:03d}.png'),
        dpi=120, bbox_inches='tight')
    plt.close('all')

    # ── Corner plot ───────────────────────────────────────────────────
    fig_corner = bayes.results.corner_plot()
    fig_corner.savefig(
        os.path.join(PATHS['corners'], f'corner_bin{i+1:03d}.png'),
        dpi=100, bbox_inches='tight')
    plt.close('all')

    # ── Integrated flux ───────────────────────────────────────────────
    flux = bayes.results.get_point_source_flux(10 * u.keV, 10 * u.MeV)
    log.info('  10 keV-10 MeV flux = %s', flux)

    # ── Collect parameters ────────────────────────────────────────────
    df  = bayes.results.get_data_frame('equal tail')
    df  = df.drop('unit', axis=1, errors='ignore')
    fit_rows.append(df.to_numpy().reshape(12))

    log.info('  Bin %d / %d complete.', i + 1, n_bins)

print(f'\nAll {n_bins} bins fitted successfully.')


12:24:58  INFO      ==== Time bin 1 / 4 ====


 *****************************************************
 MultiNest v3.10
 Copyright Farhan Feroz & Mike Hobson
 Release Jul 2015

 no. of live points =  100
 dimensionality =    3
 *****************************************************
  analysing data from chains/fit-.txt ln(ev)=  -1261.2038888465088      +/-   6.1989780487650026E-002
 Total Likelihood Evaluations:          362
 Sampling finished. Exiting MultiNest



Maximum a posteriori probability (MAP) point:

,result,unit
parameter,,
GBM.spectrum.main.Cutoff_powerlaw.K,(0.0011 +0.0024 +1.2) x 10^-2,1 / (keV s cm2)
GBM.spectrum.main.Cutoff_powerlaw.index,-1.3 +/- 0.4,
GBM.spectrum.main.Cutoff_powerlaw.xc,(2.3 -0.8 +2.8) x 10^2,keV


Values of -log(posterior) at the minimum:

...

Propagating errors:   0%|                                                                                    |…

12:25:53  WARNING   Too few points to create valid contours


processing Bayesian analyses:   0%|                                                                           …

12:25:55  INFO        10 keV-10 MeV flux =                                             flux  \
GBM: total  1.3878077147432808e-09 erg / (s cm2)   

                                       low bound  \
GBM: total  1.4227035401481844e-11 erg / (s cm2)   

                                       hi bound  
GBM: total  7.585443269771108e-09 erg / (s cm2)  
12:25:55  INFO        Bin 4 / 4 complete.



All 4 bins fitted successfully.


## Cell 11 — Save Results to CSV

In [ ]:
fit_array = np.array(fit_rows)
cutoffpl_fit_param = Table(fit_array, names=PARAM_NAMES)

# ── Attach bin edges as extra columns ────────────────────────────────
cutoffpl_fit_param['t_start'] = bb_starts
cutoffpl_fit_param['t_stop']  = bb_stops
cutoffpl_fit_param['dt']      = bb_stops - bb_starts

csv_path = os.path.join(PATHS['root'], 'cutoffpl_fit_params.csv')
cutoffpl_fit_param.write(csv_path, overwrite=True)

print(f'Results saved to: {csv_path}')
print(f'Shape: {fit_array.shape}  ({n_bins} bins x 16 parameters)')
cutoffpl_fit_param


Results saved to: gbm_pipeline_output/band_fit_params.csv
Shape: (4, 12)  (4 bins x 16 parameters)


B_K,B_K_p_err,B_K_n_err,B_K_avg_err,B_alpha,B_alpha_p_err,B_alpha_n_err,B_alpha_avg_err,E_c,E_c_p_err,E_c_n_err,E_c_avg_err,t_start,t_stop,dt
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
1.0847144193441524e-05,2.3916731774601743e-05,0.012466157706716136,0.0062450372192453685,-1.3076107437730637,-0.3988950945958165,0.4251806009798065,0.4120378477878115,229.3071204960149,-81.6566116161008,276.8984622616546,179.2775369388777,-9.999761998653412,-6.61941796541214,3.380344033241272
0.2089774472281307,0.14938696956396272,1.14908630511442,0.6492366373391913,-0.7913372445409221,-0.6024184678763067,-0.18852231557095023,0.20694807615267824,102.49696580687805,29.537642618014047,285.17876422786645,157.35820342294025,-6.61941796541214,7.057295024394989,13.676712989807129
0.9059489834046881,0.0003283511682269369,1.664418356411961,0.8323733537900939,-1.0172812822339319,-0.3213903195743948,0.0003728165036558373,0.1608815680390253,168.442743701217,0.008174814630052651,213.47245031409244,106.74031256436125,7.057295024394989,11.157336056232452,4.100041031837463
1.2292568694326541e-05,6.785548748468185e-05,0.2424779462713109,0.12127290087939778,-1.6444039535290915,-0.09950264704415845,0.6665242302540642,0.3830134386491113,363.86438517351843,-220.31240034707577,80.68240521243081,150.4974027797533,11.157336056232452,49.99896001815796,38.84162396192551


## Cell 12 — Pipeline Summary

In [20]:
print('=' * 60)
print('  GBM PIPELINE COMPLETE')
print('=' * 60)
print(f'  Trigger          : {CONFIG["trigger_name"]}')
print(f'  Detectors used   : {detectors}')
print(f'  Time bins fitted : {n_bins}')
print(f'  Bin edges (s)    : starts={bb_starts}, stops={bb_stops}')
print(f'  Output root      : {PATHS["root"]}')
print(f'  Spectra plots    : {PATHS["spectra"]}')
print(f'  Corner plots     : {PATHS["corners"]}')
print(f'  Lightcurves      : {PATHS["lc"]}')
print(f'  CSV results      : {csv_path}')
print('=' * 60)


  GBM PIPELINE COMPLETE
  Trigger          : bn140304557
  Detectors used   : ['n3', 'n4', 'b0']
  Time bins fitted : 4
  Bin edges (s)    : starts=[-9.999762   -6.61941797  7.05729502 11.15733606], stops=[-6.61941797  7.05729502 11.15733606 49.99896002]
  Output root      : gbm_pipeline_output
  Spectra plots    : gbm_pipeline_output/spectra
  Corner plots     : gbm_pipeline_output/corners
  Lightcurves      : gbm_pipeline_output/lightcurves
  CSV results      : gbm_pipeline_output/band_fit_params.csv
